# Literary Geography Exploration

Interactive exploration of the pipeline outputs for *The Great Gatsby*.

**Run the full pipeline first:**
```bash
python -m src.pipeline --config config.yaml
```

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.io import read_jsonl, read_json
from src.utils.schemas import GroundedEntity, SpatialRelation, Sample, PosteriorSummary, ConstraintModel

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Entity Overview

In [ ]:
entities = read_jsonl('../data/grounded_entities.jsonl', model=GroundedEntity)

real = [e for e in entities if e.type == 'real']
fictional = [e for e in entities if e.type == 'fictional']

print(f'Total entities: {len(entities)}')
print(f'Real: {len(real)}')
print(f'Fictional (latent): {len(fictional)}')

print('\nTop fictional entities by mention count:')
for e in sorted(fictional, key=lambda x: -x.mention_count)[:10]:
    print(f'  {e.canonical_name}: {e.mention_count} mentions')

## 2. Relation Type Distribution

In [ ]:
relations = read_jsonl('../data/relations.jsonl', model=SpatialRelation)

from collections import Counter
type_counts = Counter(r.type for r in relations)
method_counts = Counter(r.extraction_method for r in relations)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(type_counts.keys(), type_counts.values())
axes[0].set_title('Relations by Type')
axes[0].set_xlabel('Relation type')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

axes[1].pie(method_counts.values(), labels=method_counts.keys(), autopct='%1.1f%%')
axes[1].set_title('Extraction Method')

plt.tight_layout()
plt.show()
print(f'Total relations: {len(relations)}')

## 3. Posterior Distribution Summaries

In [ ]:
summaries = read_jsonl('../data/convergence/posterior_summaries.jsonl', model=PosteriorSummary)

for s in summaries:
    print(f"--- {s.name} ---")
    print(f"  Posterior mean: x={s.posterior_mean['x']:.2f} km, y={s.posterior_mean['y']:.2f} km")
    print(f"  Posterior std:  x={s.posterior_std['x']:.2f} km, y={s.posterior_std['y']:.2f} km")
    print(f"  Spatial entropy: {s.spatial_entropy:.3f}")
    print(f"  Number of modes: {s.num_modes}")
    if s.r_hat:
        print(f"  R-hat: x={s.r_hat['x']:.4f}, y={s.r_hat['y']:.4f}")
    if s.constraint_satisfaction_mean is not None:
        print(f"  Constraint satisfaction: {s.constraint_satisfaction_mean:.1%} ± {s.constraint_satisfaction_std:.1%}")
    print()

## 4. Sample Energy Trace

In [ ]:
all_samples = []
for sf in sorted(Path('../data/samples').glob('*.jsonl')):
    for s in read_jsonl(str(sf), model=Sample):
        all_samples.append(s)

energies = [s.energy for s in all_samples]

plt.figure(figsize=(12, 4))
plt.plot(energies, alpha=0.7, linewidth=0.5)
plt.xlabel('Sample index')
plt.ylabel('Energy')
plt.title('MCMC Energy Trace')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Total samples loaded: {len(all_samples)}')
print(f'Mean energy: {np.mean(energies):.4f}')
print(f'Min energy: {np.min(energies):.4f}')

## 5. Scatter Plot of Posterior Samples

In [ ]:
model = read_json('../data/constraints.json', model=ConstraintModel)

fig, ax = plt.subplots(figsize=(10, 8))

# Fixed entities
for f in model.fixed_entities:
    ax.scatter(f.x, f.y, s=200, marker='^', color='navy', zorder=5)
    ax.annotate(f.name, (f.x, f.y), xytext=(5, 5), textcoords='offset points', fontsize=8, color='navy')

# Samples for each latent entity
colors = plt.cm.Set1(np.linspace(0, 1, len(model.latent_entities)))
for idx, latent in enumerate(model.latent_entities):
    xs = [s.entities[latent.entity_id].x for s in all_samples if latent.entity_id in s.entities]
    ys = [s.entities[latent.entity_id].y for s in all_samples if latent.entity_id in s.entities]
    if xs:
        ax.scatter(xs[::10], ys[::10], s=5, alpha=0.3, color=colors[idx], label=latent.name)
        ax.scatter(np.mean(xs), np.mean(ys), s=150, marker='*', color=colors[idx], zorder=6)

ax.set_xlabel('x (km east of origin)')
ax.set_ylabel('y (km north of origin)')
ax.set_title('Posterior Samples for Fictional Entities\n(stars = posterior means, triangles = real entities)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Open Interactive Visualizations

The following files can be opened in a browser:

In [ ]:
import webbrowser

vis_files = [
    '../visualizations/constraint_graph.html',
    '../visualizations/overlay_maps/full_map.html',
]

for f in vis_files:
    p = Path(f)
    if p.exists():
        print(f'Open: {p.resolve()}')
        # Uncomment to auto-open:
        # webbrowser.open(str(p.resolve()))
    else:
        print(f'Not found: {p}')